## Social Media Post Analyzer
**Problem statement**

Brands and creators regularly publish social media posts, but understanding how a post sounds and what communication purpose it serves is not always straightforward. A post may be promotional, informative, emotional, or engagement-driven. Manual interpretation is subjective and inconsistent. The goal of this project is to build a Generative AI system that analyzes the text of a social media post and classifies its tone, intent, and communication style in a structured format using only the provided content.


In [1]:
pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.8 MB/s eta 0:00:00


In [2]:
pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [9]:
import os
from google.colab import userdata
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI #Lanchain Google GeminiAI
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from typing import List,Optional


os.environ['GOOGLE_API_KEY'] = userdata.get('api_key')

In [8]:
model_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",
                                      temperature = 0)

In [10]:
# step2 - Defining a Schema using the Pydantic modules

class SocialMediaPost(BaseModel):
  tone: str = Field(description="The tone of the Social Media Post")
  intent: str = Field(description="The intent of the Post")
  Communication_style: str = Field(description="Post type")
  summery: str = Field(description="Post Summerization")

In [12]:
parser = JsonOutputParser(pydantic_object=SocialMediaPost)

In [13]:
# Step 3 : Prompt Details
# format instructions are based on the pydantic module and json parser selected

prompt = ChatPromptTemplate.from_template(
"""
You are an Expert Understand the Social Media Post. Ensure that information is extracted from the provided Social Media Post in given format.
{format_instructions}

Social Media Post:
{social_media}
"""
).partial(format_instructions=parser.get_format_instructions())

In [14]:
Social_media = "We are excited to launch our new AI-powered learning platform next week. Stay tuned for more updates!"

In [15]:
# step 4: chain all together and invoke
# Case1:
chain = prompt | model_gemini | parser
chain.invoke({"social_media":Social_media})

{'tone': 'excited',
 'intent': 'inform',
 'Communication_style': 'announcement',
 'summery': 'The user is announcing the upcoming launch of a new AI-powered learning platform next week and is asking followers to stay tuned for more updates.'}

In [16]:
# Case2: Promotions

Social_media = "Big news! Get ready to experience our revolutionary AI learning platform launching this Monday. Early users will get exclusive premium features—don’t miss out!"

In [17]:
chain.invoke({"social_media":Social_media})

{'tone': 'Excited',
 'intent': 'Promotional',
 'Communication_style': 'Announcement',
 'summery': 'Announcing the launch of a new AI learning platform this Monday, with exclusive premium features for early users.'}

In [18]:
# Case3:

Social_media="Absolutely loving the new AI learning platform! The personalized recommendations and smooth interface make learning so much easier. Great job to the team"

In [19]:
chain.invoke({"social_media":Social_media})

{'tone': 'positive',
 'intent': 'praise',
 'Communication_style': 'testimonial',
 'summery': 'The user is expressing strong satisfaction with a new AI learning platform, highlighting its personalized recommendations and user-friendly interface as key benefits.'}

In [21]:
# Case4:
Social_media="Just started exploring this new AI learning platform. The features look promising, but the interface could be more user-friendly. Hoping for improvements soon."


In [22]:
chain.invoke({"social_media":Social_media})

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 22.019689632s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM